# SkyRecon — RDD2022 Road Damage Detection Training
**Trains YOLOv8x to detect road potholes and cracks from aerial/street view**

Expected accuracy improvement: Road Potholes ~40% → ~80%

**Estimated time: ~2-3 hours on free T4 GPU**

### What this trains:
- Road potholes
- Longitudinal cracks
- Transverse cracks
- Alligator cracks

In [ ]:
# ── Step 1: Check GPU ──────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Step 2: Install dependencies ──────────────────────────────
!pip install ultralytics>=8.3.0 -q
!pip install gdown -q

In [ ]:
# ── Step 3: Download RDD2022 dataset ──────────────────────────
# RDD2022 is hosted on Crowdworks — we use the preprocessed YOLO version
import os

!mkdir -p rdd2022_dataset

# Download preprocessed RDD2022 in YOLO format from Roboflow public dataset
# This is the standard RDD2022 dataset converted to YOLO format
!pip install roboflow -q

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_FREE_API_KEY")  # Get free key at roboflow.com

# NOTE: Go to https://universe.roboflow.com/roboflow-100/road-damage-detection-full
# Click Download → YOLOv8 format → Get API key (free)
# Replace YOUR_FREE_API_KEY above with your key

project = rf.workspace("roboflow-100").project("road-damage-detection-full")
version = project.version(2)
dataset = version.download("yolov8", location="rdd2022_dataset")

print(f'Dataset downloaded to: {dataset.location}')

In [ ]:
# ── Step 4: Check dataset structure ───────────────────────────
import os
from pathlib import Path

# Find the yaml file
yaml_files = list(Path('rdd2022_dataset').rglob('*.yaml'))
print('YAML files found:', yaml_files)

# Count images
train_imgs = list(Path('rdd2022_dataset').rglob('train/images/*.jpg'))
val_imgs   = list(Path('rdd2022_dataset').rglob('valid/images/*.jpg'))
print(f'Train images: {len(train_imgs)}')
print(f'Val images:   {len(val_imgs)}')

In [ ]:
# ── Step 5: Train ──────────────────────────────────────────────
from ultralytics import YOLO
from pathlib import Path

# Get yaml path
yaml_path = list(Path('rdd2022_dataset').rglob('*.yaml'))[0]
print(f'Using dataset: {yaml_path}')

model = YOLO('yolov8x.pt')

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    workers=4,
    device=0,
    project='runs/detect',
    name='skyrecon_rdd2022',
    exist_ok=True,
    patience=10,
    save=True,
    save_period=10,
    val=True,
    degrees=10.0,
    translate=0.1,
    scale=0.4,
    fliplr=0.5,
    flipud=0.1,
    mosaic=1.0,
    mixup=0.1,
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    cos_lr=True,
    label_smoothing=0.1,
    verbose=True,
)

print('Training complete!')
print('Best model: runs/detect/skyrecon_rdd2022/weights/best.pt')

In [ ]:
# ── Step 6: Download best.pt ───────────────────────────────────
from google.colab import files
files.download('runs/detect/skyrecon_rdd2022/weights/best.pt')
print('Save as skyrecon_rdd2022.pt in SkyRecon/backend/')

## After downloading
1. Rename to `skyrecon_rdd2022.pt`
2. Copy to `SkyRecon/backend/`
3. This model is used specifically for road pothole detection
4. Tag Amazon Q to wire it into the pipeline